# Занятие 3. Transfer Learning: Feature Extraction vs Fine-tuning

## Цели занятия:
- Освоить работу с предобученными моделями (torchvision.models)
- Изучить стратегии Transfer Learning
- Решить задачу классификации на малом датасете

---

## Шаг 1. Загрузка и подготовка данных

**Важно:** Для Transfer Learning используем датасет с малым количеством изображений.

**Примечание:** Если датасет Hymenoptera недоступен, используем подмножество CIFAR-10.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import os

# Fix random seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Data transforms for transfer learning
# IMPORTANT: Use ImageNet normalization for pretrained models!

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# Try to load Hymenoptera dataset, fallback to CIFAR-10 subset
try:
    # Download Hymenoptera dataset
    import urllib.request
    import zipfile
    
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    zip_path = "./hymenoptera_data.zip"
    
    if not os.path.exists('./hymenoptera_data'):
        print("Downloading Hymenoptera dataset...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        os.remove(zip_path)
        print("Dataset downloaded!")
    
    data_dir = './hymenoptera_data'
    image_datasets = {
        x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
        for x in ['train', 'val']
    }
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=0)
        for x in ['train', 'val']
    }
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    
    print(f"Dataset: Hymenoptera")
    print(f"Classes: {class_names}")
    print(f"Train samples: {dataset_sizes['train']}")
    print(f"Val samples: {dataset_sizes['val']}")
    
except Exception as e:
    print(f"Could not load Hymenoptera dataset: {e}")
    print("Using CIFAR-10 subset instead...")
    
    # Fallback: Use small subset of CIFAR-10
    transform_cifar = transforms.Compose([
        transforms.Resize(224),  # Resize for ResNet
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
    test_full = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)
    
    # Use only 2 classes for binary classification demo
    def filter_classes(dataset, classes=[0, 1]):  # airplane, automobile
        indices = [i for i, (_, label) in enumerate(dataset) if label in classes]
        return Subset(dataset, indices[:200])  # Small subset
    
    image_datasets = {
        'train': filter_classes(train_full),
        'val': filter_classes(test_full)
    }
    
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=0)
        for x in ['train', 'val']
    }
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = ['airplane', 'automobile']
    
    print(f"\nDataset: CIFAR-10 subset")
    print(f"Classes: {class_names}")
    print(f"Train samples: {dataset_sizes['train']}")
    print(f"Val samples: {dataset_sizes['val']}")

---
## Шаг 2. Загрузка предобученной модели

In [ ]:
# Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Print model architecture
print("ResNet18 Architecture:")
print(f"Final FC layer input features: {model.fc.in_features}")
print(f"Final FC layer output features: {model.fc.out_features}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

---
## Шаг 3. Стратегия 1: Feature Extraction (Freeze)

In [ ]:
# Create fresh model for Feature Extraction
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all parameters
for param in model_fe.parameters():
    param.requires_grad = False

# Replace the final FC layer for our number of classes
num_ftrs = model_fe.fc.in_features
model_fe.fc = nn.Linear(num_ftrs, len(class_names))

model_fe = model_fe.to(device)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f"Trainable parameters (Feature Extraction): {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
from tqdm import tqdm
import time

def train_model(model, dataloaders, criterion, optimizer, num_epochs=10, strategy_name="Model"):
    """Train model and return history."""
    since = time.time()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 30)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()
            
            running_loss = 0.0
            running_corrects = 0
            
            for inputs, labels in tqdm(dataloaders[phase], desc=phase, leave=False):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase] * 100
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
            
            print(f'{phase}: Loss={epoch_loss:.4f}, Acc={epoch_acc:.2f}%')
        
        print()
    
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.2f}%')
    
    return history, best_acc

In [ ]:
# Train Feature Extraction model
print("="*50)
print("STRATEGY 1: Feature Extraction (Freeze Backbone)")
print("="*50)

set_seed(42)
criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.SGD(model_fe.fc.parameters(), lr=0.001, momentum=0.9)

history_fe, best_acc_fe = train_model(
    model_fe, dataloaders, criterion, optimizer_fe, num_epochs=10, strategy_name="Feature Extraction"
)

---
## Шаг 4-5. Стратегия 2: Fine-tuning

In [ ]:
# Create fresh model for Fine-tuning
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace the final FC layer
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(device)

# Count trainable parameters
trainable_params_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Trainable parameters (Fine-tuning): {trainable_params_ft:,}")

# Fine-tuning with lower learning rate
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.0001, momentum=0.9)  # Lower LR!

In [ ]:
# Train Fine-tuning model
print("="*50)
print("STRATEGY 2: Fine-tuning (Full Network)")
print("="*50)

set_seed(42)
history_ft, best_acc_ft = train_model(
    model_ft, dataloaders, criterion, optimizer_ft, num_epochs=10, strategy_name="Fine-tuning"
)

---
## Шаг 6. Сравнение стратегий

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Summary
comparison = {
    'Strategy': ['Feature Extraction', 'Fine-tuning'],
    'Trainable Params': [
        sum(p.numel() for p in model_fe.parameters() if p.requires_grad),
        sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
    ],
    'Best Val Acc (%)': [best_acc_fe.item(), best_acc_ft.item()]
}

df = pd.DataFrame(comparison)
print("\n" + "="*50)
print("STRATEGY COMPARISON")
print("="*50)
print(df.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history_fe['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history_fe['val_loss'], 'b-', label='Feature Extraction', linewidth=2)
axes[0].plot(epochs, history_ft['val_loss'], 'r-', label='Fine-tuning', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Validation Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history_fe['val_acc'], 'b-', label='Feature Extraction', linewidth=2)
axes[1].plot(epochs, history_ft['val_acc'], 'r-', label='Fine-tuning', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy (%)')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transfer_learning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Попробовать разморозить все слои сети (Full Fine-tuning)
2. Сравнить время обучения и итоговую точность
3. Сохранить веса лучшей модели (.pth)

---
## Контрольные вопросы

1. Почему важно использовать нормализацию ImageNet для предобученных моделей?
2. В чем риск размораживания всех слоёв при малом датасете?
3. Почему Learning Rate для Fine-tuning должен быть меньше?
4. Как изменить количество выходных классов в ResNet?

In [ ]:
# Save best model
torch.save(model_ft.state_dict(), 'best_model_transfer_learning.pth')
print("Model saved!")